# ICD-10 Autonomous Coding: Actor-Critic-Arbiter with Reasoning Bank

Automates medical coding by having three agents debate over ICD-10 codes:
- **Actor**: Proposes ICD-10 codes from medical text using Reasoning Bank + ChromaDB
- **Critic**: Reviews and debates the Actor's codes for N configurable rounds
- **Arbiter**: Views full debate and finalizes the codes
- **Reasoning Bank**: ChromaDB store of past coding examples used as few-shot context
- **ICD-10 DB**: ChromaDB store of ICD-10 code descriptions for validation

## 0. Install Dependencies

In [ ]:
# %pip install -U langchain langchain-core langgraph langchain-openai langchain-community
# %pip install -U chromadb langfuse python-dotenv pydantic

## 1. Environment Setup & Langfuse

In [ ]:
import os
from dotenv import load_dotenv
from langfuse import get_client, observe
from langfuse.langchain import CallbackHandler

load_dotenv()

langfuse_client = get_client()
langfuse_handler = CallbackHandler()

print("✓ Langfuse initialized")

## 2. Sample ICD-10 Data + ChromaDB Setup

In [ ]:
import chromadb
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Sample ICD-10 codes across major categories
icd10_data = [
    # Cardiovascular
    {"code": "I21.0",   "description": "ST elevation myocardial infarction involving left anterior descending coronary artery", "category": "Cardiovascular"},
    {"code": "I21.19",  "description": "ST elevation myocardial infarction involving other coronary artery of inferior wall", "category": "Cardiovascular"},
    {"code": "I21.4",   "description": "Non-ST elevation myocardial infarction (NSTEMI)", "category": "Cardiovascular"},
    {"code": "I10",     "description": "Essential (primary) hypertension", "category": "Cardiovascular"},
    {"code": "I50.9",   "description": "Heart failure, unspecified", "category": "Cardiovascular"},
    {"code": "I48.0",   "description": "Paroxysmal atrial fibrillation", "category": "Cardiovascular"},
    {"code": "I25.10",  "description": "Atherosclerotic heart disease of native coronary artery without angina pectoris", "category": "Cardiovascular"},
    {"code": "I63.9",   "description": "Cerebral infarction, unspecified", "category": "Cardiovascular"},
    # Respiratory
    {"code": "J18.9",   "description": "Pneumonia, unspecified organism", "category": "Respiratory"},
    {"code": "J44.1",   "description": "Chronic obstructive pulmonary disease with acute exacerbation", "category": "Respiratory"},
    {"code": "J45.901", "description": "Unspecified asthma with acute exacerbation", "category": "Respiratory"},
    {"code": "J96.00",  "description": "Acute respiratory failure, unspecified whether with hypoxia or hypercapnia", "category": "Respiratory"},
    {"code": "J06.9",   "description": "Acute upper respiratory infection, unspecified", "category": "Respiratory"},
    # Diabetes & Endocrine
    {"code": "E11.9",   "description": "Type 2 diabetes mellitus without complications", "category": "Endocrine"},
    {"code": "E11.65",  "description": "Type 2 diabetes mellitus with hyperglycemia", "category": "Endocrine"},
    {"code": "E11.40",  "description": "Type 2 diabetes mellitus with diabetic neuropathy, unspecified", "category": "Endocrine"},
    {"code": "E11.51",  "description": "Type 2 diabetes mellitus with diabetic peripheral angiopathy without gangrene", "category": "Endocrine"},
    {"code": "E10.9",   "description": "Type 1 diabetes mellitus without complications", "category": "Endocrine"},
    {"code": "E87.1",   "description": "Hypo-osmolality and hyponatremia", "category": "Endocrine"},
    # Infections & Sepsis
    {"code": "A41.9",   "description": "Sepsis, unspecified organism", "category": "Infection"},
    {"code": "A41.51",  "description": "Sepsis due to Escherichia coli", "category": "Infection"},
    {"code": "R65.21",  "description": "Severe sepsis with septic shock", "category": "Infection"},
    {"code": "U07.1",   "description": "COVID-19", "category": "Infection"},
    {"code": "N39.0",   "description": "Urinary tract infection, site not specified", "category": "Infection"},
    # Neurological
    {"code": "G43.909", "description": "Migraine, unspecified, not intractable, without status migrainosus", "category": "Neurological"},
    {"code": "G20",     "description": "Parkinson's disease", "category": "Neurological"},
    {"code": "G30.9",   "description": "Alzheimer's disease, unspecified", "category": "Neurological"},
    {"code": "G40.909", "description": "Epilepsy, unspecified, not intractable, without status epilepticus", "category": "Neurological"},
    # Mental Health
    {"code": "F32.1",   "description": "Major depressive disorder, single episode, moderate", "category": "Mental Health"},
    {"code": "F32.2",   "description": "Major depressive disorder, single episode, severe without psychotic features", "category": "Mental Health"},
    {"code": "F41.1",   "description": "Generalized anxiety disorder", "category": "Mental Health"},
    {"code": "F20.9",   "description": "Schizophrenia, unspecified", "category": "Mental Health"},
    {"code": "F10.20",  "description": "Alcohol dependence, uncomplicated", "category": "Mental Health"},
    # Musculoskeletal
    {"code": "M54.5",   "description": "Low back pain", "category": "Musculoskeletal"},
    {"code": "M17.11",  "description": "Primary osteoarthritis, right knee", "category": "Musculoskeletal"},
    {"code": "M06.9",   "description": "Rheumatoid arthritis, unspecified", "category": "Musculoskeletal"},
    # Fractures
    {"code": "S72.001A","description": "Fracture of unspecified part of neck of right femur, initial encounter for closed fracture", "category": "Injury"},
    {"code": "S52.501A","description": "Unspecified fracture of the lower end of right radius, initial encounter", "category": "Injury"},
    # GI
    {"code": "K92.1",   "description": "Melena", "category": "Gastrointestinal"},
    {"code": "K57.30",  "description": "Diverticulosis of large intestine without perforation or abscess without bleeding", "category": "Gastrointestinal"},
    {"code": "K74.60",  "description": "Unspecified cirrhosis of liver", "category": "Gastrointestinal"},
    {"code": "K29.70",  "description": "Gastritis, unspecified, without bleeding", "category": "Gastrointestinal"},
    # Renal
    {"code": "N18.3",   "description": "Chronic kidney disease, stage 3 (moderate)", "category": "Renal"},
    {"code": "N17.9",   "description": "Acute kidney failure, unspecified", "category": "Renal"},
    # Cancer
    {"code": "C34.10",  "description": "Malignant neoplasm of upper lobe, unspecified bronchus or lung", "category": "Oncology"},
    {"code": "C50.911", "description": "Malignant neoplasm of unspecified site of right female breast", "category": "Oncology"},
    {"code": "C18.9",   "description": "Malignant neoplasm of colon, unspecified", "category": "Oncology"},
    # Symptoms
    {"code": "R07.9",   "description": "Chest pain, unspecified", "category": "Symptoms"},
    {"code": "R55",     "description": "Syncope and collapse", "category": "Symptoms"},
    {"code": "R06.09",  "description": "Other forms of dyspnea", "category": "Symptoms"},
    {"code": "R41.3",   "description": "Other amnesia", "category": "Symptoms"},
    {"code": "R73.09",  "description": "Other abnormal glucose", "category": "Symptoms"},
]

# Load into ChromaDB
icd10_docs = [
    Document(
        page_content=f"{d['code']}: {d['description']}",
        metadata={"code": d["code"], "category": d["category"]}
    )
    for d in icd10_data
]

icd10_vectorstore = Chroma.from_documents(
    documents=icd10_docs,
    embedding=embeddings,
    collection_name="icd10_codes"
)
icd10_retriever = icd10_vectorstore.as_retriever(search_kwargs={"k": 8})

print(f"✓ ICD-10 ChromaDB loaded with {len(icd10_docs)} codes")

## 3. Reasoning Bank Setup

The Reasoning Bank stores past successful coding cases. When the Actor receives new medical text, it retrieves the top-3 most similar past cases and uses them as few-shot examples in its prompt. This grounds the Actor's reasoning in proven coding patterns.

In [ ]:
# Reasoning Bank: past coding examples with rationale and final agreed codes
reasoning_bank_data = [
    {
        "medical_text": "67-year-old male with crushing chest pain radiating to left arm, diaphoresis, EKG shows ST elevation in leads II, III, aVF. Troponin 2.4. Known hypertension.",
        "actor_rationale": "ST elevation in inferior leads indicates inferior STEMI. Hypertension is a comorbidity.",
        "critic_feedback": "Agreed. Inferior STEMI correctly identified. Hypertension appropriately coded as secondary.",
        "final_codes": ["I21.19", "I10"],
        "final_descriptions": ["STEMI inferior wall", "Essential hypertension"],
        "confidence": "high"
    },
    {
        "medical_text": "52-year-old female with Type 2 diabetes, HbA1c 10.2%, tingling and numbness in both feet for 6 months. No wounds.",
        "actor_rationale": "Type 2 DM with peripheral neuropathy. Elevated HbA1c indicates hyperglycemia.",
        "critic_feedback": "Both neuropathy and hyperglycemia codes needed. HbA1c of 10.2% confirms poor control.",
        "final_codes": ["E11.40", "E11.65"],
        "final_descriptions": ["T2DM with diabetic neuropathy", "T2DM with hyperglycemia"],
        "confidence": "high"
    },
    {
        "medical_text": "78-year-old with COPD, worsening dyspnea, increased sputum, fever 38.8C. CXR shows right lower lobe infiltrate.",
        "actor_rationale": "COPD exacerbation triggered by pneumonia. Both should be coded separately.",
        "critic_feedback": "Correct. COPD exacerbation and pneumonia are distinct and both coded. Order: COPD first as primary reason for visit.",
        "final_codes": ["J44.1", "J18.9"],
        "final_descriptions": ["COPD with acute exacerbation", "Pneumonia unspecified organism"],
        "confidence": "high"
    },
    {
        "medical_text": "35-year-old with persistent low mood, anhedonia, insomnia, fatigue for 3 months. PHQ-9 score 14. No prior episodes.",
        "actor_rationale": "PHQ-9 of 14 = moderate severity. Single episode based on history.",
        "critic_feedback": "Correct. Moderate severity aligns with PHQ-9 14-19 range. Single episode confirmed.",
        "final_codes": ["F32.1"],
        "final_descriptions": ["MDD single episode moderate"],
        "confidence": "high"
    },
    {
        "medical_text": "Patient unresponsive, blood cultures positive for E. coli, temp 39.5C, WBC 18k, BP 85/50 despite 2L IVF bolus.",
        "actor_rationale": "E. coli sepsis with refractory hypotension despite fluids = septic shock.",
        "critic_feedback": "Both sepsis and septic shock codes required. Sepsis due to E. coli coded specifically, not unspecified.",
        "final_codes": ["A41.51", "R65.21"],
        "final_descriptions": ["Sepsis due to E. coli", "Severe sepsis with septic shock"],
        "confidence": "high"
    },
    {
        "medical_text": "45-year-old female admitted for elective right knee replacement. History of primary osteoarthritis right knee, well-controlled hypertension.",
        "actor_rationale": "Primary osteoarthritis right knee is the reason for surgery. Hypertension is comorbidity.",
        "critic_feedback": "Correct principal diagnosis. Hypertension as secondary. No additional codes needed.",
        "final_codes": ["M17.11", "I10"],
        "final_descriptions": ["Primary osteoarthritis right knee", "Essential hypertension"],
        "confidence": "high"
    },
    {
        "medical_text": "82-year-old female fell at home, hip X-ray shows displaced femoral neck fracture right side. First fracture, no osteoporosis documented.",
        "actor_rationale": "Displaced femoral neck fracture right hip, initial encounter. 7th character A for initial encounter.",
        "critic_feedback": "Correct. 7th character A appropriate for initial encounter. Consider if osteoporosis should be queried.",
        "final_codes": ["S72.001A"],
        "final_descriptions": ["Fracture femoral neck right initial encounter"],
        "confidence": "high"
    },
    {
        "medical_text": "Patient with known cirrhosis presents with hematemesis, melena. Upper endoscopy shows esophageal varices with active bleeding.",
        "actor_rationale": "Cirrhosis with esophageal varices bleeding. GI bleed manifested as melena.",
        "critic_feedback": "Cirrhosis with bleeding varices has a specific combination code. Melena is integral to the bleed and may not need separate coding.",
        "final_codes": ["K74.60", "K92.1"],
        "final_descriptions": ["Unspecified cirrhosis of liver", "Melena"],
        "confidence": "medium"
    },
]

# Store reasoning bank in ChromaDB
reasoning_docs = [
    Document(
        page_content=entry["medical_text"],
        metadata={
            "final_codes": ", ".join(entry["final_codes"]),
            "final_descriptions": ", ".join(entry["final_descriptions"]),
            "actor_rationale": entry["actor_rationale"],
            "critic_feedback": entry["critic_feedback"],
            "confidence": entry["confidence"]
        }
    )
    for entry in reasoning_bank_data
]

reasoning_vectorstore = Chroma.from_documents(
    documents=reasoning_docs,
    embedding=embeddings,
    collection_name="reasoning_bank"
)
reasoning_retriever = reasoning_vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✓ Reasoning Bank loaded with {len(reasoning_docs)} past coding examples")

## 4. State Definition

In [ ]:
from typing import TypedDict, Annotated, List
import operator

class ICD10CodingState(TypedDict):
    medical_text: str          # Input medical note
    max_rounds: int            # Configurable debate rounds (default 3)
    round_number: int          # Current round
    actor_codes: List[str]     # Latest codes proposed by Actor
    actor_rationale: str       # Actor's reasoning
    critic_feedback: str       # Critic's latest feedback
    critic_agrees: bool        # Whether Critic has accepted the codes
    debate_history: Annotated[list, operator.add]  # Full debate transcript
    final_codes: List[str]     # Arbiter's final codes
    arbiter_rationale: str     # Arbiter's explanation
    arbiter_confidence: str    # high / medium / low

print("✓ ICD10CodingState defined")

## 5. Actor Agent

The Actor retrieves similar cases from the **Reasoning Bank** and relevant ICD-10 codes from **ChromaDB**, then proposes codes with rationale.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import json

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

actor_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert medical coder specializing in ICD-10-CM coding.
Your job is to propose accurate ICD-10 codes for the given medical text.

Rules:
- Code to the highest level of specificity
- List the principal diagnosis first
- Include all relevant comorbidities
- Use valid ICD-10-CM codes only
- If this is a revision round, address the Critic's feedback specifically

Respond in this exact JSON format:
{{
  "proposed_codes": ["CODE1", "CODE2"],
  "rationale": "Your detailed reasoning here",
  "response_to_critic": "How you addressed the critic's feedback (empty string if first round)"
}}"""),
    ("user", """Medical Text:
{medical_text}

Similar Past Cases from Reasoning Bank:
{reasoning_examples}

Relevant ICD-10 Codes from Database:
{icd10_candidates}

Critic's Previous Feedback (empty if first round):
{critic_feedback}

Current Round: {round_number} of {max_rounds}

Propose your ICD-10 codes:""")
])

@observe(name="actor-agent")
def actor_node(state: ICD10CodingState) -> dict:
    # Retrieve from Reasoning Bank
    reasoning_docs = reasoning_retriever.invoke(state["medical_text"])
    reasoning_examples = "\n\n".join([
        f"Example {i+1}:\n"
        f"  Text: {doc.page_content}\n"
        f"  Final Codes: {doc.metadata['final_codes']}\n"
        f"  Rationale: {doc.metadata['actor_rationale']}"
        for i, doc in enumerate(reasoning_docs)
    ])

    # Retrieve relevant ICD-10 codes
    icd10_docs = icd10_retriever.invoke(state["medical_text"])
    icd10_candidates = "\n".join([doc.page_content for doc in icd10_docs])

    chain = actor_prompt | llm
    result = chain.invoke({
        "medical_text": state["medical_text"],
        "reasoning_examples": reasoning_examples,
        "icd10_candidates": icd10_candidates,
        "critic_feedback": state.get("critic_feedback", ""),
        "round_number": state["round_number"] + 1,
        "max_rounds": state["max_rounds"]
    }, config={"callbacks": [langfuse_handler]})

    try:
        parsed = json.loads(result.content)
    except:
        parsed = {"proposed_codes": [], "rationale": result.content, "response_to_critic": ""}

    round_num = state["round_number"] + 1
    debate_entry = {
        "round": round_num,
        "actor": {
            "codes": parsed["proposed_codes"],
            "rationale": parsed["rationale"],
            "response_to_critic": parsed.get("response_to_critic", "")
        }
    }

    print(f"\n[Round {round_num}] Actor proposes: {parsed['proposed_codes']}")

    return {
        "actor_codes": parsed["proposed_codes"],
        "actor_rationale": parsed["rationale"],
        "round_number": round_num,
        "debate_history": [debate_entry]
    }

print("✓ Actor agent defined")

## 6. Critic Agent

The Critic reviews the Actor's proposed codes against the ICD-10 database and coding guidelines. After N rounds it must make a final agree/disagree decision.

In [ ]:
critic_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a senior medical coding auditor with expertise in ICD-10-CM guidelines.
Your job is to critically review proposed ICD-10 codes for accuracy and completeness.

Review criteria:
- Are codes valid and at the highest specificity?
- Is the principal diagnosis correctly identified?
- Are any codes missing or incorrect?
- Do codes follow ICD-10-CM official guidelines?
- Are combination codes used where applicable?

Respond in this exact JSON format:
{{
  "agrees": true/false,
  "feedback": "Detailed feedback on the proposed codes",
  "suggested_changes": ["specific change 1", "specific change 2"],
  "verdict": "ACCEPT" or "REVISE"
}}"""),
    ("user", """Medical Text:
{medical_text}

Actor's Proposed Codes: {actor_codes}
Actor's Rationale: {actor_rationale}

Relevant ICD-10 Descriptions for Validation:
{icd10_candidates}

Similar Past Cases from Reasoning Bank:
{reasoning_examples}

Round {round_number} of {max_rounds}.
{'This is the FINAL round — you must make a definitive ACCEPT or REVISE decision.' if round_number >= max_rounds else 'You may request revisions if needed.'}

Review the proposed codes:""")
])

@observe(name="critic-agent")
def critic_node(state: ICD10CodingState) -> dict:
    # Retrieve from Reasoning Bank for validation
    reasoning_docs = reasoning_retriever.invoke(state["medical_text"])
    reasoning_examples = "\n\n".join([
        f"Example {i+1}:\n"
        f"  Text: {doc.page_content}\n"
        f"  Final Codes: {doc.metadata['final_codes']}\n"
        f"  Critic Note: {doc.metadata['critic_feedback']}"
        for i, doc in enumerate(reasoning_docs)
    ])

    icd10_docs = icd10_retriever.invoke(" ".join(state["actor_codes"]) + " " + state["medical_text"])
    icd10_candidates = "\n".join([doc.page_content for doc in icd10_docs])

    chain = critic_prompt | llm
    result = chain.invoke({
        "medical_text": state["medical_text"],
        "actor_codes": state["actor_codes"],
        "actor_rationale": state["actor_rationale"],
        "icd10_candidates": icd10_candidates,
        "reasoning_examples": reasoning_examples,
        "round_number": state["round_number"],
        "max_rounds": state["max_rounds"]
    }, config={"callbacks": [langfuse_handler]})

    try:
        parsed = json.loads(result.content)
    except:
        parsed = {"agrees": True, "feedback": result.content, "suggested_changes": [], "verdict": "ACCEPT"}

    # Update last debate entry with critic response
    critic_entry = {
        "round": state["round_number"],
        "critic": {
            "agrees": parsed["agrees"],
            "feedback": parsed["feedback"],
            "suggested_changes": parsed.get("suggested_changes", []),
            "verdict": parsed.get("verdict", "REVISE")
        }
    }

    verdict = parsed.get("verdict", "REVISE")
    print(f"[Round {state['round_number']}] Critic verdict: {verdict}")
    if not parsed["agrees"]:
        print(f"  Changes requested: {parsed.get('suggested_changes', [])}")

    return {
        "critic_feedback": parsed["feedback"],
        "critic_agrees": parsed["agrees"],
        "debate_history": [critic_entry]
    }

print("✓ Critic agent defined")

## 7. Arbiter Agent

The Arbiter reviews the entire debate transcript and makes the final coding decision. It can side with the Actor, the Critic, or propose a reconciled set of codes.

In [ ]:
arbiter_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a chief medical coding officer with final authority on ICD-10 code assignments.
You have reviewed the Actor-Critic debate and must make the final coding decision.

Your decision should:
- Be based on the full debate context
- Select the most clinically accurate and compliant codes
- Reconcile valid points from both Actor and Critic
- Provide a confidence level for audit purposes

Respond in this exact JSON format:
{{
  "final_codes": ["CODE1", "CODE2"],
  "rationale": "Why these codes are correct",
  "sided_with": "Actor" or "Critic" or "Reconciled",
  "confidence": "high" or "medium" or "low",
  "audit_note": "Any flags or notes for human review"
}}"""),
    ("user", """Medical Text:
{medical_text}

Full Debate History:
{debate_history}

Final Actor Codes: {actor_codes}
Critic's Final Position: {critic_agrees} (True=Agreed, False=Still disagreed)

Make your final coding decision:""")
])

@observe(name="arbiter-agent")
def arbiter_node(state: ICD10CodingState) -> dict:
    debate_str = json.dumps(state["debate_history"], indent=2)

    chain = arbiter_prompt | llm
    result = chain.invoke({
        "medical_text": state["medical_text"],
        "debate_history": debate_str,
        "actor_codes": state["actor_codes"],
        "critic_agrees": state["critic_agrees"]
    }, config={"callbacks": [langfuse_handler]})

    try:
        parsed = json.loads(result.content)
    except:
        parsed = {
            "final_codes": state["actor_codes"],
            "rationale": result.content,
            "sided_with": "Actor",
            "confidence": "medium",
            "audit_note": ""
        }

    print(f"\n[Arbiter] Final codes: {parsed['final_codes']}")
    print(f"[Arbiter] Sided with: {parsed['sided_with']} | Confidence: {parsed['confidence']}")
    if parsed.get("audit_note"):
        print(f"[Arbiter] Audit note: {parsed['audit_note']}")

    return {
        "final_codes": parsed["final_codes"],
        "arbiter_rationale": parsed["rationale"],
        "arbiter_confidence": parsed["confidence"]
    }

print("✓ Arbiter agent defined")

## 8. Build LangGraph with Debate Loop

In [ ]:
from langgraph.graph import StateGraph, END, START

def should_continue(state: ICD10CodingState) -> str:
    """Route: continue debating or send to arbiter."""
    if state["critic_agrees"]:
        print(f"\n→ Critic agreed at round {state['round_number']} — sending to Arbiter")
        return "arbiter"
    if state["round_number"] >= state["max_rounds"]:
        print(f"\n→ Max rounds ({state['max_rounds']}) reached — sending to Arbiter")
        return "arbiter"
    print(f"→ Critic disagrees — starting round {state['round_number'] + 1}")
    return "actor"

builder = StateGraph(ICD10CodingState)
builder.add_node("actor", actor_node)
builder.add_node("critic", critic_node)
builder.add_node("arbiter", arbiter_node)

builder.add_edge(START, "actor")
builder.add_edge("actor", "critic")
builder.add_conditional_edges("critic", should_continue, {"actor": "actor", "arbiter": "arbiter"})
builder.add_edge("arbiter", END)

coding_graph = builder.compile()

print("✓ LangGraph compiled")
print("  Flow: START → Actor → Critic → [debate loop] → Arbiter → END")

## 9. Run ICD-10 Coding Pipeline

In [ ]:
@observe(name="icd10-coding-pipeline")
def run_coding_pipeline(medical_text: str, max_rounds: int = 3) -> dict:
    """Run the full Actor-Critic-Arbiter coding pipeline."""
    langfuse_client.set_current_trace_io(input={"medical_text": medical_text, "max_rounds": max_rounds})

    initial_state = {
        "medical_text": medical_text,
        "max_rounds": max_rounds,
        "round_number": 0,
        "actor_codes": [],
        "actor_rationale": "",
        "critic_feedback": "",
        "critic_agrees": False,
        "debate_history": [],
        "final_codes": [],
        "arbiter_rationale": "",
        "arbiter_confidence": ""
    }

    result = coding_graph.invoke(initial_state)
    langfuse_client.set_current_trace_io(output={"final_codes": result["final_codes"], "confidence": result["arbiter_confidence"]})
    langfuse_client.flush()

    return result


# Test cases
test_cases = [
    {
        "label": "Cardiac",
        "text": "58-year-old male presents to ED with acute onset chest pressure, shortness of breath. EKG shows new LBBB. Troponin I elevated at 3.1 ng/mL. History of hypertension and Type 2 diabetes. Started on heparin drip."
    },
    {
        "label": "Sepsis",
        "text": "72-year-old female nursing home resident brought in with altered mental status, fever 39.2C, WBC 22,000, UA positive for bacteria and WBCs. Blood pressure 88/52 requiring vasopressors. Blood cultures pending."
    },
    {
        "label": "Psychiatric",
        "text": "28-year-old male with no prior psychiatric history presenting with 6 weeks of depressed mood, inability to work, passive suicidal ideation without plan, poor sleep and appetite. PHQ-9 score 18."
    }
]

results = []
for case in test_cases:
    print(f"\n{'='*65}")
    print(f"CASE: {case['label']}")
    print(f"{'='*65}")
    print(f"Text: {case['text'][:100]}...")

    result = run_coding_pipeline(case["text"], max_rounds=3)

    print(f"\n{'─'*65}")
    print(f"FINAL CODES  : {result['final_codes']}")
    print(f"CONFIDENCE   : {result['arbiter_confidence']}")
    print(f"ROUNDS TAKEN : {result['round_number']}")
    print(f"RATIONALE    : {result['arbiter_rationale'][:200]}...")

    results.append({"case": case["label"], **result})

## 10. Summary

In [ ]:
print("\n" + "="*65)
print("CODING SUMMARY")
print("="*65)
for r in results:
    print(f"\n{r['case']}")
    print(f"  Final Codes : {r['final_codes']}")
    print(f"  Confidence  : {r['arbiter_confidence']}")
    print(f"  Rounds      : {r['round_number']} of {r['max_rounds']}")

print("\n✓ View full traces at: https://us.cloud.langfuse.com")
print("  Search trace name: icd10-coding-pipeline")